# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a transparent rule-based review queue for the Content Refresh Prioritization decision. The feature window is February 2026, and no future-window outcomes or leakage-prone fields are used.

## 0. Setup and data access

This notebook uses the February 2026 partition of the FlyRank warehouse. The Hugging Face read token is stored in Colab Secrets as `HF_TOKEN` and is never written directly into the notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded:", HF_TOKEN is not None)

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# IMPORTANT: ML-07 uses February features, so the source partition must also be February.
TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
}

print("Warehouse connection configured for February 2026.")

## 1. My rule and its reason codes

### Baseline rule

I will prioritize existing content pages that had meaningful Google Search visibility in February but showed relatively weak click-through performance for that level of exposure.

The score combines search visibility volume with February click-through performance. It is a transparent prioritization rule, not a prediction of future traffic.

### Reason codes

- `high_visibility_low_ctr` — substantial February search exposure with below-benchmark CTR.
- `high_visibility` — substantial February search exposure without below-benchmark CTR.
- `limited_visibility` — low February search exposure, so the recommendation is lower confidence.

In [ ]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS feb_avg_position,
        SUM(ga4_sessions) AS feb_sessions,
        SUM(scroll_events) AS feb_scroll_events
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

In [ ]:
print("Columns:")
print(feature_frame.columns.tolist())

print("\nMissing values:")
print(feature_frame.isna().sum())

print("\nDuplicate client-content pairs:", feature_frame.duplicated(["client_hash_id", "content_hash_id"]).sum())

assert len(feature_frame) > 0, "Feature frame is empty; check the February warehouse partition and date filter."

### Signal Check 1 — Search Visibility Volume

**Signal:** February Google Search impressions.

**Hypothesis:** Pages with more February impressions have more observable search exposure. A minimum-volume threshold helps avoid making strong prioritization decisions from very small impression counts.

I will inspect impression buckets and their average CTR before deciding how visibility volume should affect the baseline score.

In [ ]:
signal_df = feature_frame.copy()

signal_df["feb_ctr"] = np.where(
    signal_df["feb_impressions"] > 0,
    signal_df["feb_clicks"] / signal_df["feb_impressions"],
    np.nan
)
signal_df["feb_ctr_pct"] = signal_df["feb_ctr"] * 100

signal_df["impression_bucket"] = pd.cut(
    signal_df["feb_impressions"],
    bins=[-1, 99, 499, 999, 4999, np.inf],
    labels=["<100", "100–499", "500–999", "1,000–4,999", "5,000+"]
)

bucket_summary = (
    signal_df.groupby("impression_bucket", observed=False)
    .agg(
        rows=("content_hash_id", "size"),
        avg_impressions=("feb_impressions", "mean"),
        avg_ctr_pct=("feb_ctr_pct", "mean")
    )
    .reset_index()
)

display(bucket_summary)

In [ ]:
non_empty_buckets = bucket_summary[bucket_summary["rows"] > 0].copy()
high_volume_rows = int(signal_df["feb_impressions"].ge(100).sum())

if len(non_empty_buckets) >= 2 and high_volume_rows > 0:
    signal_verdict = "CONFIRMED"
else:
    signal_verdict = "MIXED"

print("Signal Check 1 verdict:", signal_verdict)
print("Pages with at least 100 February impressions:", high_volume_rows)

### Signal Check 1 Verdict

The code above reports the verdict from the observed February data. I use impression volume as a visibility-strength component of the baseline, while keeping the claim directional: higher exposure means a larger observable opportunity, not guaranteed future improvement.

## 2. Build the ranked queue (writes the CSV)

The score is intentionally simple: meaningful search visibility is combined with the gap between the page's February CTR and the benchmark CTR. Larger visibility plus a larger negative CTR gap produces higher review priority.

In [ ]:
baseline_df = signal_df.copy()

baseline_df["visibility_eligible"] = baseline_df["feb_impressions"] >= 100

ctr_benchmark = baseline_df.loc[
    baseline_df["visibility_eligible"] & baseline_df["feb_ctr_pct"].notna(),
    "feb_ctr_pct"
].median()

print("CTR benchmark (%):", ctr_benchmark)
assert pd.notna(ctr_benchmark), "CTR benchmark is undefined; check February data and visibility threshold."

In [ ]:
baseline_df["visibility_score"] = np.log1p(baseline_df["feb_impressions"])
baseline_df["ctr_gap"] = (ctr_benchmark - baseline_df["feb_ctr_pct"]).clip(lower=0)
baseline_df["action_score"] = baseline_df["visibility_score"] * baseline_df["ctr_gap"]

baseline_df["reason_code"] = np.select(
    [
        baseline_df["visibility_eligible"] & (baseline_df["feb_ctr_pct"] < ctr_benchmark),
        baseline_df["visibility_eligible"]
    ],
    ["high_visibility_low_ctr", "high_visibility"],
    default="limited_visibility"
)

baseline_df["action"] = np.select(
    [
        baseline_df["reason_code"] == "high_visibility_low_ctr",
        baseline_df["reason_code"] == "high_visibility"
    ],
    ["Review first", "Review next"],
    default="Lower-priority review"
)

ranked_queue = (
    baseline_df.sort_values(
        ["action_score", "feb_impressions"],
        ascending=[False, False],
        na_position="last"
    ).reset_index(drop=True)
)
ranked_queue["rank"] = ranked_queue.index + 1

ranked_queue.head(20)

In [ ]:
output_columns = [
    "rank", "client_hash_id", "content_hash_id",
    "action_score", "action", "reason_code",
    "feb_impressions", "feb_clicks", "feb_ctr_pct",
    "feb_avg_position", "feb_sessions", "feb_scroll_events"
]

ranked_output = ranked_queue[output_columns].copy()

os.makedirs("work/outputs", exist_ok=True)
repo_output_path = "work/outputs/baseline_action_score.csv"
ranked_output.to_csv(repo_output_path, index=False)

print("Saved:", repo_output_path)
print("Rows:", len(ranked_output))

## 3. Top-20 review

The following table contains the 20 highest-ranked pages. Each row has an action and reason code. Confidence is higher when impression volume is substantial because the CTR signal is based on more observed search exposure.

A recommendation can still be wrong because this baseline does not capture content quality, search intent alignment, seasonality, technical issues, or business context.

In [ ]:
top20_review = ranked_output.head(20).copy()
top20_review

### Top-20 review notes

For manual review, pay particular attention to high-impression pages whose score is driven by a very large CTR gap. These are useful candidates but can be weak picks if the observed CTR reflects a legitimate search-intent or brand effect rather than a refresh opportunity.

In [ ]:
print("Lowest-scoring items within the Top-20:")
display(top20_review.tail(5))

## 4. Weak picks + leakage check

The weakest-looking Top-20 items should be treated as review candidates rather than automatic refresh decisions.

### Leakage check

The ranking uses only February feature-window fields: impressions, clicks, derived CTR, average position, sessions, and scroll events. It does not use future-window outcomes, decline labels, `trend_direction`, `trend_pct`, or product decision fields.

In [ ]:
feature_columns_used = [
    "feb_impressions", "feb_clicks", "feb_ctr_pct",
    "feb_avg_position", "feb_sessions", "feb_scroll_events"
]

forbidden_terms = ["label", "trend", "future", "outcome", "decision", "product"]
leakage_candidates = [
    col for col in feature_columns_used
    if any(term in col.lower() for term in forbidden_terms)
]

print("Feature columns used:")
print(feature_columns_used)
print("Potential leakage candidates:", leakage_candidates)
assert leakage_candidates == [], "Unexpected leakage-related feature detected."

In [ ]:
required_columns = [
    "rank", "client_hash_id", "content_hash_id",
    "action_score", "action", "reason_code"
]

assert all(column in ranked_output.columns for column in required_columns)
assert len(ranked_output) > 0
assert len(top20_review) == min(20, len(ranked_output))
assert ranked_output.duplicated(["client_hash_id", "content_hash_id"]).sum() == 0

print("Total ranked rows:", len(ranked_output))
print("Top-20 rows:", len(top20_review))
print("Duplicate client-content pairs:", ranked_output.duplicated(["client_hash_id", "content_hash_id"]).sum())
print("\nReason-code counts:")
print(ranked_output["reason_code"].value_counts())
print("\nAction counts:")
print(ranked_output["action"].value_counts())

## Self-check

- [x] Every section is filled with markdown reasoning and supporting code.
- [ ] Run the notebook top to bottom with Runtime → Run all and confirm all assertions pass.
- [x] Claims are framed as observed, directional, and decision-support rather than guaranteed outcomes.
- [x] No client names, URLs, or private search queries are exposed.
- [ ] Confirm `work/outputs/baseline_action_score.csv` contains the non-empty ranked queue before final submission.